# 01 Deep EDA – NESO Historic Demand Data

This notebook performs exploratory data analysis before modelling.

## A. Project context

This EDA phase is designed to identify the best demand target variable, right time frequency for modelling, main seasonal patterns, missingness and data quality issues, candidate external variables for later SARIMAX modelling, anomalies and shock periods, and possible assumptions for later scenario simulation. No forecasting models are built here.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.utils import RAW_DATA_DIR, TABLES_DIR, EDA_FIGURES_DIR, load_json
from src import eda

sns.set_theme(style="whitegrid")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
EDA_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## B. Load raw NESO data

Load the combined raw file documented in `data/raw/selected_resource_info.json`, standardise column names and inspect the basic structure.

In [ ]:
selected_info_path = RAW_DATA_DIR / "selected_resource_info.json"
if not selected_info_path.exists():
    raise FileNotFoundError("Run `python src/ingest_neso.py` before this notebook, or manually set `raw_data_path`.")

selected_info = load_json(selected_info_path)
raw_data_path = PROJECT_ROOT / selected_info["combined_output_path"]
print(raw_data_path)

raw_df = eda.load_raw_data(raw_data_path)
df = eda.standardise_column_names(raw_df)
print("Shape:", df.shape)
display(df.head())
display(df.tail())
display(df.dtypes.to_frame("dtype"))

## C. Dataset structure

In [ ]:
column_summary = eda.summarise_dataframe(df)
missing_summary = eda.summarise_missing_values(df)
datetime_candidates = eda.detect_datetime_columns(df)
numeric_columns = eda.identify_numeric_columns(df)
categorical_columns = [c for c in df.columns if c not in numeric_columns]

column_summary.to_csv(TABLES_DIR / "column_summary.csv", index=False)
missing_summary.to_csv(TABLES_DIR / "missing_values_summary.csv", index=False)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Datetime candidates:", datetime_candidates)
print("Numeric columns:", numeric_columns)
print("Categorical/non-numeric columns:", categorical_columns)
print("Duplicate rows:", df.duplicated().sum())
display(column_summary)
display(missing_summary)

In [ ]:
datetime_column = datetime_candidates[0] if datetime_candidates else None
if datetime_column is None:
    raise ValueError("No datetime column detected. Inspect columns and set `datetime_column` manually.")

df = eda.parse_datetime_column(df, datetime_column)
print("Selected datetime column:", datetime_column)
print("Date range:", df[datetime_column].min(), "to", df[datetime_column].max())
frequency_info = eda.infer_time_frequency(df, datetime_column)
print(frequency_info)
duplicate_timestamps = eda.find_duplicate_timestamps(df, datetime_column)
print("Duplicate timestamps:", len(duplicate_timestamps))

## D. Candidate target variable analysis

In [ ]:
candidate_targets = eda.identify_candidate_demand_columns(df)
print("Candidate demand target columns:", candidate_targets)

target_rows = []
for column in candidate_targets:
    series = df[column]
    target_rows.append({
        "column": column,
        "count": series.count(),
        "missing_count": series.isna().sum(),
        "missing_pct": series.isna().mean() * 100,
        "mean": series.mean(),
        "std": series.std(),
        "min": series.min(),
        "median": series.median(),
        "max": series.max(),
    })
    eda.plot_time_series(df, datetime_column, column, EDA_FIGURES_DIR / f"time_series_{column}.png")
    eda.plot_distribution(df, column, EDA_FIGURES_DIR / f"distribution_{column}.png")

candidate_target_summary = pd.DataFrame(target_rows)
candidate_target_summary.to_csv(TABLES_DIR / "candidate_target_summary.csv", index=False)
display(candidate_target_summary)

target_column = candidate_targets[0] if candidate_targets else numeric_columns[0]
print("Provisional recommended target variable:", target_column)

## E. Time-series pattern analysis

In [ ]:
eda.plot_time_series(df, datetime_column, target_column, EDA_FIGURES_DIR / "recommended_target_full_series.png")
recent = df.sort_values(datetime_column).tail(min(len(df), 5000))
eda.plot_time_series(recent, datetime_column, target_column, EDA_FIGURES_DIR / "recommended_target_recent_series.png")
eda.plot_daily_pattern_if_applicable(df, datetime_column, target_column, EDA_FIGURES_DIR / "daily_pattern.png")
eda.plot_day_of_week_pattern(df, datetime_column, target_column, EDA_FIGURES_DIR / "day_of_week_pattern.png")
eda.plot_monthly_pattern(df, datetime_column, target_column, EDA_FIGURES_DIR / "monthly_pattern.png")
eda.plot_yearly_pattern(df, datetime_column, target_column, EDA_FIGURES_DIR / "yearly_pattern.png")

pattern_df = df[[datetime_column, target_column]].dropna().copy()
pattern_df["is_weekend"] = pattern_df[datetime_column].dt.dayofweek >= 5
display(pattern_df.groupby("is_weekend")[target_column].describe())

## F. Missingness and data quality

In [ ]:
eda.plot_missing_values(df, EDA_FIGURES_DIR / "missing_values_matrix.png")
print("Missing values saved to outputs/tables/missing_values_summary.csv")
print("Duplicate timestamps:", len(duplicate_timestamps))

non_positive = df[df[target_column] <= 0] if target_column in df else pd.DataFrame()
print("Suspicious zero or negative target values:", len(non_positive))

# Timestamp gaps based on the most common interval.
sorted_times = df[datetime_column].dropna().sort_values().drop_duplicates()
common_interval = sorted_times.diff().dropna().mode()
if not common_interval.empty:
    expected = pd.date_range(sorted_times.min(), sorted_times.max(), freq=common_interval.iloc[0])
    missing_timestamps = expected.difference(sorted_times)
    print("Expected timestamps:", len(expected))
    print("Missing timestamps in sequence:", len(missing_timestamps))

## G. Outliers and shocks

In [ ]:
outliers = eda.detect_outliers_iqr(df, target_column)
outlier_summary = outliers[[datetime_column, target_column]].sort_values(target_column) if not outliers.empty else pd.DataFrame(columns=[datetime_column, target_column])
outlier_summary.to_csv(TABLES_DIR / "outlier_summary.csv", index=False)
print("IQR outliers:", len(outliers))
display(outlier_summary.head(20))
display(outlier_summary.tail(20))

if df[datetime_column].dt.year.min() <= 2020 <= df[datetime_column].dt.year.max():
    covid_period = df[df[datetime_column].between("2020-03-01", "2021-12-31")]
    print("COVID-era rows available for visual inspection:", len(covid_period))

## H. External variable analysis

In [ ]:
external_candidates = [c for c in numeric_columns if c != target_column]
correlation_table = eda.compute_correlation_table(df, target_column)
correlation_table.to_csv(TABLES_DIR / "external_variable_correlation.csv", index=False)
display(correlation_table.head(30))
eda.plot_correlation_heatmap(df, EDA_FIGURES_DIR / "correlation_heatmap.png")

for column in correlation_table["column"].head(5):
    ax = df.sort_values(datetime_column).set_index(datetime_column)[[target_column, column]].dropna().tail(2000).plot(figsize=(14, 5), title=f"{target_column} and {column}")
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(EDA_FIGURES_DIR / f"target_vs_{column}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

lag_rows = []
for column in correlation_table["column"].head(10):
    for lag in [1, 2, 24, 48]:
        lag_rows.append({"column": column, "lag": lag, "lagged_correlation": df[target_column].corr(df[column].shift(lag))})
lag_correlation = pd.DataFrame(lag_rows)
display(lag_correlation)

## I. Stationarity and decomposition

In [ ]:
stationarity = eda.run_stationarity_checks(df, datetime_column, target_column)
print(stationarity)

rolling = df[[datetime_column, target_column]].dropna().sort_values(datetime_column).set_index(datetime_column)[target_column]
window = stationarity.get("rolling_window", 48)
ax = rolling.rolling(window).mean().plot(figsize=(14, 5), label="rolling mean")
rolling.rolling(window).std().plot(ax=ax, label="rolling std")
ax.legend()
ax.set_title(f"Rolling mean and standard deviation for {target_column}")
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(EDA_FIGURES_DIR / "rolling_mean_std.png", dpi=150, bbox_inches="tight")
plt.close(fig)

decomposition_result = eda.decompose_time_series(df, datetime_column, target_column, EDA_FIGURES_DIR / "seasonal_decomposition.png")
print(decomposition_result)

## J. EDA conclusions

Use this section after running the notebook to record: recommended target variable, modelling frequency, train/test split, rolling-origin validation approach, key seasonality, candidate SARIMAX regressors, whether Prophet may be useful later, whether deep learning is necessary, scenario simulation assumptions and a capacity-threshold approach.

Provisional guidance: start with transparent baselines and daily or native-frequency aggregation after confirming missingness and timestamp quality. Deep learning models are not necessary at this stage unless later baselines prove inadequate and enough clean history is available.